In [ ]:
%pip install shap

In [ ]:
import shap
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForMaskedLM

# 1. Load the model and tokenizer
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)
model.eval()

# 2. Define the individual data point to analyze
n1, n2 = "virus", "infection"
prompt = f"A {n1} {n2} refers to an {n2} that <mask> a {n1}."
# Assume we want to analyze why the model predicts “causes” 
target_word = " causes" 
target_id = tokenizer.convert_tokens_to_ids(target_word)

# 3. prediction function
def mlm_predict_score(texts):
    """
    Input: A list of text strings perturbed by SHAP (e.g., [“A virus infection refers...”, “A virus [PAD] refers...”])
    Output: The logit score for predicting the target_word at the <mask> position
    """
    scores = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt")
        
        # find index of <mask> token
        # Mask token id of RoBERTa is tokenizer.mask_token_id (50264)
        mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
        
        # If the text is so distorted that even the mask is gone, give it a very low score
        if len(mask_token_index) == 0:
            scores.append(-100.0)
            continue
            
        with torch.no_grad():
            outputs = model(**inputs)
        
        # Get the logits for all words at the mask position
        mask_logits = outputs.logits[0, mask_token_index, :]
        
        # Calculate the score for the target word (e.g., “causes”)
        target_score = mask_logits[0, target_id].item()
        scores.append(target_score)
        
    return np.array(scores)

# 4. Initialize SHAP Explainer
explainer = shap.Explainer(mlm_predict_score, tokenizer)

# 5. SHAP analysis 
shap_values = explainer([prompt])

# 6. Visualization
# This generates a text-based visualization, where red indicates a positive contribution and blue indicates a negative contribution
shap.plots.text(shap_values[0])

In [ ]:
import shap
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForMaskedLM

# 1. load the model
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)
model.eval()

# 2. Load our failure cases: 
# Relation: attribute
# ------------------------------------------------------------
# Compound   : incinerator plant
# Prompt     : An incinerator plant refers to a plant that <mask> incinerator.
# Predictions: ['produces', 'uses', 'processes', 'manufactures', 'burns']
# ------------------------------------------------------------

n1, n2 = "incinerator", "plant"
# original Prompt 
prompt = f"An {n1} {n2} refers to a {n2} that <mask> {n1}."

# 3. Define 【True Positive】 and 【False Positive】
correct_word = " features" 
wrong_word = " produces" 

correct_id = tokenizer(correct_word, add_special_tokens=False)["input_ids"][0]
wrong_id = tokenizer(wrong_word, add_special_tokens=False)["input_ids"][0]

print(f"Correct ID: {correct_id}, Wrong ID: {wrong_id}")

def mlm_predict_scores(texts):
    scores_correct = []
    scores_wrong = []
    
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt")
        # find <mask_token> 
        mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
        
        if len(mask_token_index) == 0:
            scores_correct.append(-100.0)
            scores_wrong.append(-100.0)
            continue
            
        with torch.no_grad():
            outputs = model(**inputs)
        
        # get scores
        mask_logits = outputs.logits[0, mask_token_index[0], :]
        scores_correct.append(mask_logits[correct_id].item())
        scores_wrong.append(mask_logits[wrong_id].item())
        
    return np.stack([scores_correct, scores_wrong], axis=-1)

# 4. SHAP analysis
explainer = shap.Explainer(mlm_predict_scores, tokenizer)
shap_values = explainer([prompt])

# ==========================================
# 5. Visualization
# ==========================================

# fig 1. which words are “blocking” the model's predict features
plt.figure(figsize=(10, 5))
shap.plots.bar(shap_values[0, :, 0], show=False)
plt.title("Why the model DID NOT predict 'features' (Correct)")
plt.savefig("shap_correct_features.png", bbox_inches="tight")
plt.close()

# fig 2. which words are “influencing” the model's predict produces
plt.figure(figsize=(10, 5))
shap.plots.bar(shap_values[0, :, 1], show=False)
plt.title("Why the model PREDICTED 'produces' (Wrong)")
plt.savefig("shap_wrong_produces.png", bbox_inches="tight")
plt.close()

print("please check figures：shap_correct_features.png and shap_wrong_produces.png")

In [ ]:
import shap
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForMaskedLM

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)
model.eval()

# Relation: attribute
# ------------------------------------------------------------
# Compound   : incinerator plant
# Prompt     : An incinerator plant refers to a plant that <mask> incinerator.
# Predictions: ['produces', 'uses', 'processes', 'manufactures', 'burns']
# ------------------------------------------------------------

n1, n2 = "incinerator", "plant"
# add article 'an'
prompt = "An incinerator plant refers to a plant that <mask> an incinerator."

# ‘features’ as the representative term for the attribute relationship;  ‘produces’ as the term for model prediction errors
correct_word = " features" 
wrong_word = " produces" 

correct_id = tokenizer(correct_word, add_special_tokens=False)["input_ids"][0]
wrong_id = tokenizer(wrong_word, add_special_tokens=False)["input_ids"][0]

def mlm_predict_scores(texts):
    scores_correct = []
    scores_wrong = []
    
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt")
        mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
        
        if len(mask_token_index) == 0:
            scores_correct.append(-100.0)
            scores_wrong.append(-100.0)
            continue
            
        with torch.no_grad():
            outputs = model(**inputs)
        
        
        mask_logits = outputs.logits[0, mask_token_index[0], :]
        scores_correct.append(mask_logits[correct_id].item())
        scores_wrong.append(mask_logits[wrong_id].item())
        
    return np.stack([scores_correct, scores_wrong], axis=-1)

# 4. SHAP analysis
explainer = shap.Explainer(mlm_predict_scores, tokenizer)
shap_values = explainer([prompt])

# ==========================================
# 5. visualization
# ==========================================

# which words are “blocking” the model's predict features
plt.figure(figsize=(10, 5))
shap.plots.bar(shap_values[0, :, 0], show=False)
plt.title("Why the model DID NOT predict 'features' (Correct) with an")
plt.savefig("shap_correct_features_with_an.png", bbox_inches="tight")
plt.close()

# which words are “influencing” the model's predict produces
plt.figure(figsize=(10, 5))
shap.plots.bar(shap_values[0, :, 1], show=False)
plt.title("Why the model PREDICTED 'produces' (Wrong) with an")
plt.savefig("shap_wrong_produces_with_an.png", bbox_inches="tight")
plt.close()

print("please see the figures：shap_correct_features.png and shap_wrong_produces.png")
